# Pokedex Project: 
---

### Opening Question
**What distinguishes legendary Pokemon from non‑legendary ones? Is it just higher stats, or do certain types and physical traits also play a role?**

### Objective
To explore the Pokemon dataset and identify the key characteristics that separate legendary from non‑legendary species. I'll be examining type distributions, stat differences, generational trends, and type combinations. Then, build a logistic regression model to predict legendary status and interpret which features are most influential.

### Dataset Overview
- **Sample Size**: 800+ Pokemon from multiple generations
- **Variables**: name, type1, type2, hp, attack, defence, sp_attack, sp_defence, speed, base_total, capture_rate, base_happiness, weight_kg, height_m, generation, is_legendary, and more
- **Analysis Approach**: Exploratory Data Analysis [EDA] followed by logistic regression modelling.

In [ ]:
# import libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# visualisation style
sns.set_style("whitegrid")

# colour scheme for consistency
primary = "#d59de3"
secondary = "#345882"
success = "#2ca02c"
danger = "#d62728"
gradient = ["#d59de3", "#b291d6", "#8c84c7", "#6477b6", "#48689d", "#345882"]

In [ ]:
# colour mapping
typecolours = {
    "normal": "#c4c4b8",
    "fire": "#ffa06a",
    "water": "#7bafff",
    "grass": "#8ed66f",
    "electric": "#ffe26b",
    "ice": "#b5eae7",
    "fighting": "#dc6b66",
    "poison": "#be73bc",
    "ground": "#e8d08d",
    "flying": "#c1b3f9",
    "psychic": "#fa82a7",
    "bug": "#c2d16c",
    "rock": "#d1c37b",
    "ghost": "#9d8bb9",
    "dragon": "#9973ff",
    "dark": "#8e7d73",
    "steel": "#cacad8",
    "fairy": "#f5aedb",
    "none": "#ba9ce258"
}

### 1. Data Loading and Viewing
Before beginning the EDA, I need the dataset to be present first, of course. Here's a brief look into what to expect from the data.

In [ ]:
# load dataset
path = "pokemon.csv"
df = pd.read_csv(path)

# shape and types
print(f"Shape of the dataset: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

In [ ]:
print("First 5 records:")
df.head()

In [ ]:
print("Last 5 records:")
df.tail()

In [ ]:
print("Descriptive statistics:")
df.describe(include="all")

In [ ]:
# data quality check
print("Missing values:")
missingdata = df.isnull().sum()
missingpct = (missingdata / len(df) * 100).round(2)
print(pd.DataFrame({"Missing Values": missingdata, "Percentage": missingpct}))
print("Duplicate values: ",df.duplicated().sum())

### 2. Types Analysis
To start off the EDA, I'll be analysing the primary and secondary types of the Pokemon present in the dataset. Also mention how these types are proportioned, and check whether a primary type can be a secondary type.

In [ ]:
# distribution of pokemon types
type1 = df["type1"].value_counts()
type1pct = (type1 / type1.sum() * 100).round(2)

print("Distribution of Pokemon by Type1:")
pd.DataFrame({"Count": type1, "Percentage": type1pct}) 

In [ ]:
# visualisation for type1
colours1 = [typecolours[t] for t in type1pct.index]

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=type1.values, y=type1.index, palette=colours1, ax=ax)
ax.set_title("Distribution of Pokemon by Type1", fontsize=15, fontweight="bold")
ax.set_xlabel("Count")
ax.set_ylabel("Type")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
_, txtlabels, autotxt = plt.pie(type1pct, labels=type1pct.index, labeldistance = 1.05, autopct='%1.1f%%', startangle=478, pctdistance=0.78, colors=colours1, explode = [0.02] * len(type1pct), wedgeprops={"edgecolor": "white", "linewidth": 1})
centre = plt.Circle((0, 0), 0.6, fc="white")
ax.add_artist(centre)
for txt1 in txtlabels:
    txt1.set_fontsize(10)
for txt2 in autotxt:
    txt2.set_fontsize(7)
ax.set_title("Type1 Proportions", fontsize=15, fontweight="bold", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
type2 = df["type2"].fillna("none").value_counts() # to properly show the null values present
type2pct = (type2 / type2.sum() * 100).round(2)

print("Distribution of Pokemon by Type2:")
pd.DataFrame({"Count": type2, "Percentage": type2pct})

In [ ]:
# visualisation for type2
colours2 = [typecolours[t] for t in type2pct.index]

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=type2.values, y=type2.index, palette=colours2, ax=ax)
ax.set_title("Distribution of Pokemon by Type2", fontsize=15, fontweight="bold")
ax.set_xlabel("Count")
ax.set_ylabel("Type")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
_, txtlabels, autotxt = plt.pie(type2pct, labels=type2pct.index, labeldistance = 1.05, autopct='%1.1f%%', pctdistance=0.78, startangle=47, colors=colours2, explode = [0.02] * len(type2pct), wedgeprops={"edgecolor": "white", "linewidth": 1})
centre = plt.Circle((0, 0), 0.6, fc="white")
ax.add_artist(centre)
for txt1 in txtlabels:
    txt1.set_fontsize(10)
for txt2 in autotxt:
    txt2.set_fontsize(7)
ax.set_title("Type2 Proportions", fontsize=15, fontweight="bold", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# overlapping types
overlap = pd.DataFrame({"Type1": type1, "Type2": type2}).dropna()
overlap["Total"] = overlap["Type1"] + overlap["Type2"]
overlap["% of Overlap"] = ((overlap["Total"] / overlap["Total"].sum()) * 100).round(2)
overlap

### 3. Pokemon Classification: Single vs Dual Type

Here I check if a Pokemon has a secondary type. The majority are dual‑type, and I also examine how base total compares between the two classes.

In [ ]:
# create pokemon class
df["poke_class"] = df["type2"].apply(lambda x: "Dual" if pd.notna(x) else "Single")
classct = df["poke_class"].value_counts()

print("Pokemon class distribution:")
print(classct)
print(f"\nPercentages:")
print((classct / len(df) * 100).round(2))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=classct.index, y=classct.values, palette=gradient, ax=ax)
ax.set_title("Single vs Dual Type Pokemon", fontsize=14, fontweight="bold")
ax.set_xlabel("Pokemon Class", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
for i, v in enumerate(classct.values):
    ax.text(i, v + 1, f"{v:,}", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nAverage base total by class:")
print(df.groupby("poke_class")["base_total"].agg(["mean", "median"]).round(2))

### 4. Generation Analysis

I look at how many Pokemon appear per generation and what proportion of them are legendary. Later generations tend to introduce more legendaries.

In [ ]:
# generation distribution
gen_counts = df["generation"].value_counts().sort_index()
print("Generation distribution:")
print(gen_counts)
print(f"\nPercentages:")
print((gen_counts / len(df) * 100).round(2))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=gen_counts.index, y=gen_counts.values, palette="crest", ax=ax)
ax.set_title("Pokemon Distribution by Generation", fontsize=14, fontweight="bold")
ax.set_xlabel("Generation", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
for i, v in enumerate(gen_counts.values):
    ax.text(i, v + 1, f"{v:,}", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

# legendary by generation
legendary_by_gen = pd.crosstab(df["generation"], df["is_legendary"])
print("Legendary distribution by generation:")
print(legendary_by_gen)

legendary_by_gen_pct = pd.crosstab(df["generation"], df["is_legendary"], normalize="index") * 100
print("\nPercentage legendary by generation:")
print(legendary_by_gen_pct.round(2))

### 5. Stat Distribution Analysis

Legendary Pokemon have significantly higher stats across the board. The boxplots below show the clear separation between the two groups.

In [ ]:
stat_cols = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed", "base_total"]

print("Average stats by legendary status:")
stat_comparison = df.groupby("is_legendary")[stat_cols].mean().round(2)
stat_comparison["count"] = df.groupby("is_legendary").size()
print(stat_comparison)

# boxplot for each stat
for col in stat_cols:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=df, x="is_legendary", y=col, palette=gradient, ax=ax)
    ax.set_title(f"{col.capitalize()} Distribution by Legendary Status", fontsize=14, fontweight="bold")
    ax.set_xlabel("Legendary", fontsize=12)
    ax.set_ylabel(col.capitalize(), fontsize=12)
    ax.set_xticklabels(["No", "Yes"])
    plt.tight_layout()
    plt.show()

### 6. Correlation Analysis

The correlation matrix reveals that base total and special attack have the strongest positive correlation with legendary status, while capture rate has a strong negative correlation [legendary Pokemon are harder to catch].

In [ ]:
df["capture_rate"] = pd.to_numeric(df["capture_rate"], errors="coerce")
numeric_cols = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed", "base_total", "capture_rate", "base_happiness", "weight_kg", "height_m", "generation", "is_legendary"]
correlation_matrix = df[numeric_cols].corr()
print("Correlation matrix:")
correlation_matrix.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".3f", center=0, square=True, linewidths=1, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlation Matrix of Key Variables", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 7. Type vs Legendary Relationship

Certain types are much more likely to be legendary. The bar chart below shows the percentage of legendary Pokemon for each primary type.

In [ ]:
# type vs legendary
legendary = pd.crosstab(df["type1"], df["is_legendary"], margins=True)
print("Type1 vs Legendary distribution:")
print(legendary)

legendarypct = pd.crosstab(df["type1"], df["is_legendary"], normalize="index") * 100
print("\nPercentage legendary by type:")
print(legendarypct.round(2))

# filter for types with at least 10 pokemon for readability
typect = df["type1"].value_counts()
filteredlegendary = legendarypct.loc[typect[typect >= 10].index]

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=filteredlegendary[1], y=filteredlegendary.index, palette="viridis", ax=ax)
ax.set_title("Legendary Percentage by Primary Type (types with 10+ Pokemon)", fontsize=14, fontweight="bold")
ax.set_xlabel("Percentage Legendary", fontsize=12)
ax.set_ylabel("Type", fontsize=12)
plt.tight_layout()
plt.show()

### 8. Type Combinations
What combinations are the most common? That's the question I aim to answer here.

In [ ]:
type2cleaned = df["type2"].fillna("none")
df["type_combination"] = df["type1"] + (" + " + type2cleaned)
combct = df["type_combination"].value_counts() # count of each combination

top10 = combct.head(10)
print("Top 10 type combinations:")
top10

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top10.values, y=top10.index, palette="mako", ax=ax)
ax.set_title("Top 10 Type Combinations", fontsize=14, fontweight="bold")
ax.set_xlabel("Count", fontsize=12)
ax.set_ylabel("Type Combination", fontsize=12)
plt.tight_layout()
plt.show()

### 9. Top Performers by Base Total
The strongest Pokemon are almost all legendary, reinforcing the idea that legendary status is closely tied to raw power.

In [ ]:
top15 = df.nlargest(15, "base_total")[["name", "type1", "type2", "base_total", "is_legendary"]]
print("Top 15 Pokemon by Base Total:")
print(top15)

high_performers = df[df["base_total"] >= 600]
print(f"\nPokemon with base total 600+: {len(high_performers)}")
print(f"Legendary percentage: {high_performers['is_legendary'].mean() * 100:.1f}%")
print(f"Type distribution:\n{high_performers['type1'].value_counts()}")

### 10. Machine Learning: Predicting Legendary Status
Here onwards, I'll proceed to build a logistic regression model to predict whether a Pokemon is legendary. Logistic regression gives interpretable coefficients, to observe exactly which features increase the odds of being legendary.

I'm using stats, capture rate, generation, and one‑hot encoded type columns as predictors.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve

print("ML libraries imported.")

In [ ]:
# prepare features and target
ml_df = df.copy()

# fill missing type2 with 'None' for encoding
ml_df["type2"] = ml_df["type2"].fillna("none")
ml_df["capture_rate"] = pd.to_numeric(ml_df["capture_rate"], errors="coerce")
ml_df = ml_df.dropna(subset=["weight_kg", "capture_rate", "base_happiness", "height_m"])
# select features
features = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed", "base_total", "capture_rate", "base_happiness", "weight_kg", "height_m", "generation", "type1", "type2"]
target = "is_legendary"

x = ml_df[features]
y = ml_df[target]

# one-hot encode type columns
x = pd.get_dummies(x, columns=["type1", "type2"], drop_first=False)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {len(x_train)} Pokemon")
print(f"Test set: {len(x_test)} Pokemon")
print(f"Legendary percentage in training: {y_train.mean() * 100:.1f}%")
print(f"Legendary percentage in test: {y_test.mean() * 100:.1f}%")

In [ ]:
# train logistic regression model
model = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
model.fit(x_train, y_train)

# predictions
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)[:, 1]

# evaluation
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("Model performance on test set [in %]:")
print(f"Accuracy: {acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall: {rec * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")
print(f"ROC-AUC: {auc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Non-legendary", "Legendary"]))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="viridis", xticklabels=["Non-legendary", "Legendary"], yticklabels=["Non-legendary", "Legendary"], linewidth=0.5, ax=ax)
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba) # fpr: false pos, tpr: true pos, _: thresholds, which is basically cutoff vals for prediction probabilities

fig, ax = plt.subplots(figsize=(8, 6)) 
ax.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.4f})", color=secondary) # auc: area under curve, measures how well model can distinguish between classes [legendary/non-legendary]
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
ax.set_title("ROC Curve", fontsize=14, fontweight="bold")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# feature importance from coefficients
coef_df = pd.DataFrame({"Feature": x.columns, "Coefficient": model.coef_[0]})
coef_df["abs_coef"] = coef_df["Coefficient"].abs()
coef_df = coef_df.reset_index(drop=True).head(15)

print("Top 15 most influential features:")
coef_df[["Feature", "Coefficient"]]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x="Coefficient", y="Feature", data=coef_df, palette="viridis", ax=ax)
ax.axvline(0, color="black", linestyle="--")
ax.set_title("Top 15 Features for Predicting Legendary Status", fontsize=14, fontweight="bold")
ax.set_xlabel("Logistic Regression Coefficient", fontsize=12)
ax.set_ylabel("Feature", fontsize=12)
plt.tight_layout()
plt.show()

# why -1 to 1?
#  coefs in a logistic regression can be interpreted as follows: if a coef has pos value, it means it increases chances of pokemon to be legendary. if neg, then it reduces chances of being legendary.

In [ ]:
# cross-validation
cv_scores = cross_val_score(model, x_train, y_train, cv=5, scoring="accuracy")
print(f"\nCross-validation accuracy scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean() * 100:.2f}%")
print(f"Standard deviation: {cv_scores.std() * 100:.2f}%")

In [ ]:
import joblib
# saving the model
joblib.dump(model, "pokemon_legendary_model.pkl")
print("Model saved as 'pokemon_legendary_model.pkl'.")

### Machine Learning Takeaway

The model achieved the following performance on the test data:
* Accuracy: 94.23%
* Precision: 61.90%
* Recall: 92.86%
* F1 Score: 74.29%
* ROC-AUC: 97.18%
* Mean Cross-Validation Accuracy: 92.47%

These results show that logistic regression can very effectively identify legendary Pokemon. The high ROC-AUC indicates excellent discrimination between the two classes.

The most influential features are special attack, base total, and capture rate [lower capture rate = rarer]. Types such as Psychic and Dragon also have strong positive coefficients, confirming what was seen in the EDA.

### Conclusions and Observations

Legendary Pokemon have significantly higher stats across the board, with special attack being the most distinguishing feature. They are often Psychic, Dragon, or Fire types and have very low capture rates.

Most Pokemon are dual‑type [over 50%], but being dual‑type does not guarantee higher stats. Legendary Pokemon are more common in later generations [especially generations 3‑5] and are generally heavier than non‑legendary species.

The logistic regression model confirms the visual findings: base total, special attack, and type are the strongest predictors of legendary status. The model's high accuracy suggests these features capture the essential differences.